In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [88]:
# Format all floats to 6 decimal places with commas
pd.set_option("display.float_format", "{:.2f}".format)

In [89]:
df = pd.read_csv(r'C:\Users\Stride-Dev05\Downloads\Books\Python Pandas Portfolio\Subscription Breakeven Analysis - Stratascratch\dropbox_subscription_plans.csv')

In [90]:
df.shape

(16, 7)

In [91]:
df.dtypes

plan_id                    int64
plan_name                 object
segment                   object
fixed_costs                int64
variable_cost_per_user     int64
revenue_per_subscriber     int64
current_subscribers        int64
dtype: object

In [92]:
df.isna().sum()

plan_id                   0
plan_name                 0
segment                   0
fixed_costs               0
variable_cost_per_user    0
revenue_per_subscriber    0
current_subscribers       0
dtype: int64

In [93]:
df.duplicated().sum()

0

In [94]:
df.head()

,plan_id,plan_name,segment,fixed_costs,variable_cost_per_user,revenue_per_subscriber,current_subscribers
0,301,Basic,Consumer,12500,3,10,4000
1,302,Plus,Consumer,40000,6,25,3000
2,303,Premium,Consumer,90000,12,50,1500
3,304,Family,Consumer,45000,9,30,2000
4,305,Starter,Consumer,22000,5,14,3500


In [95]:
## Total Count of Subscribers per plan
df.groupby('plan_name')['current_subscribers'].sum().sort_values(ascending=False)

plan_name
Business      6000
Education     5000
Team          5000
Basic         4000
Pro           4000
Enterprise    3500
Starter       3500
Plus          3000
Elite         2800
Student       2500
Family        2000
Legacy        2000
Premium       1500
Promo         1500
Nonprofit     1200
On-Prem        500
Name: current_subscribers, dtype: int64

In [96]:
df['total_revenue'] = df['revenue_per_subscriber'] * df['current_subscribers']
df['total_variable_costs'] = df['variable_cost_per_user'] * df['current_subscribers']
df['total_costs'] = df['fixed_costs'] + df['total_variable_costs']
df['profit'] = df['total_revenue'] - df['total_costs']

In [97]:
df.head(16)

,plan_id,plan_name,segment,fixed_costs,variable_cost_per_user,revenue_per_subscriber,current_subscribers,total_revenue,total_variable_costs,total_costs,profit
0,301,Basic,Consumer,12500,3,10,4000,40000,12000,24500,15500
1,302,Plus,Consumer,40000,6,25,3000,75000,18000,58000,17000
2,303,Premium,Consumer,90000,12,50,1500,75000,18000,108000,-33000
3,304,Family,Consumer,45000,9,30,2000,60000,18000,63000,-3000
4,305,Starter,Consumer,22000,5,14,3500,49000,17500,39500,9500
5,306,Elite,Consumer,130000,20,75,2800,210000,56000,186000,24000
6,307,Team,Business,120000,18,60,5000,300000,90000,210000,90000
7,308,Business,Business,250000,35,120,6000,720000,210000,460000,260000
8,309,Pro,Business,180000,28,90,4000,360000,112000,292000,68000
9,310,Enterprise,Business,600000,90,300,3500,1050000,315000,915000,135000


In [98]:
## Total Count of Subscribers and total revenue per plan
df.groupby('plan_name').agg({
'current_subscribers' : 'sum',
'total_revenue':'sum'    
}).sort_values('total_revenue',ascending=False)

,current_subscribers,total_revenue
plan_name,,
Enterprise,3500,1050000
Business,6000,720000
Pro,4000,360000
Team,5000,300000
Elite,2800,210000
On-Prem,500,110000
Plus,3000,75000
Premium,1500,75000
Family,2000,60000


In [99]:
#The breakeven point for each plan (the number of subscribers it needs before it stops losing money). 
# Breakeven point is  where they do not make any profit or any loss -> breakeven point = fixed costs/ selling price per item - variable costs per item
# basically the breakeven subscribers value will give us an idea about how many users we need to sell the subscripton plan to  initally cover the total costs before making any actual revenue/profits.

df['breakeven_subscribers'] = (df['fixed_costs'] )/ (df['revenue_per_subscriber'] - df['variable_cost_per_user'])
df['breakeven_subscribers'] = np.where ((df['revenue_per_subscriber'] - df['variable_cost_per_user']) == 0 ,np.NaN,df['breakeven_subscribers'] )

## Calculated these paramets to cross verify the breakeven subscribers count .
df['breakeven_revenue'] = df['breakeven_subscribers'] * df['revenue_per_subscriber']
df['breakeven_total_costs'] = df['fixed_costs'] + (df['variable_cost_per_user'] * df['breakeven_subscribers'])
df['breakeven_profit'] = df['breakeven_revenue'] - df['breakeven_total_costs']
df['breakeven_profit'] =df['breakeven_profit'].mask(df['breakeven_profit'].abs() < 1e-7,0.0)
df[['plan_id','breakeven_subscribers','breakeven_revenue','breakeven_total_costs','breakeven_profit']]

,plan_id,breakeven_subscribers,breakeven_revenue,breakeven_total_costs,breakeven_profit
0,301,1785.71,17857.14,17857.14,0.00
1,302,2105.26,52631.58,52631.58,0.00
2,303,2368.42,118421.05,118421.05,0.00
3,304,2142.86,64285.71,64285.71,0.00
4,305,2444.44,34222.22,34222.22,0.00
5,306,2363.64,177272.73,177272.73,0.00
6,307,2857.14,171428.57,171428.57,0.00
7,308,2941.18,352941.18,352941.18,0.00
8,309,2903.23,261290.32,261290.32,0.00
9,310,2857.14,857142.86,857142.86,0.00


In [108]:
## Total Count of  Breakeven Subscribers for each plan
df.groupby('plan_name')['breakeven_subscribers'].sum().round().reset_index()

,plan_name,breakeven_subscribers
0,Basic,1786.00
1,Business,2941.00
2,Education,3750.00
3,Elite,2364.00
4,Enterprise,2857.00
5,Family,2143.00
6,Legacy,-2250.00
7,Nonprofit,1800.00
8,On-Prem,250000.00
9,Plus,2105.00


In [110]:
#How each plan's profit changes as its subscriber count grows
df.groupby(['plan_name','current_subscribers'])['profit'].sum().sort_values(ascending=False).reset_index()

,plan_name,current_subscribers,profit
0,Business,6000,260000
1,Enterprise,3500,135000
2,Team,5000,90000
3,Pro,4000,68000
4,Elite,2800,24000
5,Plus,3000,17000
6,Basic,4000,15500
7,Starter,3500,9500
8,Education,5000,5000
9,Student,2500,4000


In [119]:
# To analyse the profits of each plan as the subscriber count grows , I am going to introduce a new subscribers variabe where we add 500 more subscribers to every current subscriber count and then compare the change in profits to see the trend
# This scenario is hypothetical and an assumption to answer the second deliverable.
df['new_subscribers'] = df['current_subscribers'] + 500
df['new_subscriber_revenue'] = df['revenue_per_subscriber']*df['new_subscribers']
df['new_subscriber_costs'] = df['fixed_costs'] + (df['variable_cost_per_user'] * df['new_subscribers'] )
df['new_subscriber_profit'] = df['new_subscriber_revenue'] - df['new_subscriber_costs'] 
df['profit_difference'] = df['new_subscriber_profit'] - df['profit'] 


In [121]:
## Change in profit for each plan as the subsciber count grows 
df['profit_change'] = (df['new_subscriber_profit'] - df['profit'] ) * 100 / df['profit']
df['profit_change'] = df['profit_change'].abs()
df[['plan_name','current_subscribers','new_subscribers','profit','new_subscriber_profit','profit_difference','profit_change']]

,plan_name,current_subscribers,new_subscribers,profit,new_subscriber_profit,profit_difference,profit_change
0,Basic,4000,4500,15500,19000,3500,22.58
1,Plus,3000,3500,17000,26500,9500,55.88
2,Premium,1500,2000,-33000,-14000,19000,57.58
3,Family,2000,2500,-3000,7500,10500,350.00
4,Starter,3500,4000,9500,14000,4500,47.37
5,Elite,2800,3300,24000,51500,27500,114.58
6,Team,5000,5500,90000,111000,21000,23.33
7,Business,6000,6500,260000,302500,42500,16.35
8,Pro,4000,4500,68000,99000,31000,45.59
9,Enterprise,3500,4000,135000,240000,105000,77.78


To analyse how the profit of each plan changes as the subscriber count grows, I added 500 subscribers to the current subscriber count of each plan and
recalculated the profit.The analysis shows that most plans experience an increase in profit when 500 additional subscribers are added. 
For example, the Enterprise plans profit increases by $105,000, while the Business plans profit increases by $42,500.Some plans that are currently
operating at a loss also improve with additional subscribers. The Family plan moves from a $3,000 loss to a $7,500 profit, 
while the Nonprofit plan improves from a $3,000 loss to a $500 loss.

However, not all plans benefit from the additional subscribers. The Promo plan shows no change in profit, while the Legacy plans profit decreases 
by $2,000. The On-Prem plan only improves by $1,000 and remains at a significant loss.

Overall, based on the above analysis increasing subscriber numbers can improve profitability for most plans, 
but the impact varies depending on each plans revenue and variable costs.


The three plans that I would like  closely look at are Promo , Legacy and On-Prem plans 



#### Promo Plan 

No matter how many subscribers this plan gains, the profit will remain at the same level because the revenue per subscriber and 
variable cost per user are the same(8). This means that each additional subscriber generates revenue that is completely offset by the variable cost,
leaving no contribution toward the plans fixed costs.
To make this plan profitable, the subscriptions team would need to either increase the revenue per subscriber or decrease the variable cost per user. 
The fixed costs should also be reviewed to determine whether there are opportunities to reduce them


#### Legacy Plan

The profits tend to decrease when more subscribers are added. This is because the variable cost per user (9) is higher than the revenue per subscriber (5).i.e for every subscriber added we are generating a 4 dollar loss.  To ensure that this plan generates profits, we need to reduce the variable cost per user.


#### On-Prem Plan

The fixed costs for this plan are 500,000  and the variable cost per user is 218, while the revenue per subscriber is 220. The contribution is just $2, which is very minimal when compared to the fixed costs. the plan needs 250,000 subscribers just to break even . 
To ensure that this plan generates profits, we need to make sure that the revenue per subscriber is substantially increased or the variable cost per user is reduced  to help offset the fixed costs.

